# RankResume Stage 1 Question Selection

This notebook selects or reuses exactly six questions from the existing `bankresume` table. It does not generate the later LLM questions.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'Question_Selection_Generation':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
JD_ID = 3

from Question_Selection_Generation.db_helper import count_question_bank, ensure_schema, transaction
from Question_Selection_Generation.exam_workflow import select_questions
print(f'Project root: {PROJECT_ROOT}')
print(f'JD_ID: {JD_ID}')

Project root: c:\Python_ML\Project
JD_ID: 3


## Load configuration, schema, and Question Bank

In [2]:
with transaction() as connection:
    ensure_schema(connection)
    bank_count = count_question_bank(connection)
print(f'Question Bank rows: {bank_count}')
if bank_count == 0:
    raise RuntimeError('bankresume is empty; load the existing Question Bank before selecting questions')

Question Bank rows: 5194


## Select or reuse Stage 1

Set `REFRESH = True` only for an intentional replacement after checking that the exam has not started and has no submitted answers.

In [3]:
REFRESH = False
with transaction() as connection:
    result = select_questions(connection, jd_id=JD_ID, refresh=REFRESH)

print(f'Exam session: {result.exam_id}')
print('Reused existing selection:', result.reused)
print('Core skills:', result.classification.core_skills)
print('Changeable skills:', result.classification.changeable_skills)
print('Unclassified skills:', result.classification.unclassified_skills)

Exam session: a324098c-787f-4497-af4b-81c7be6789a0
Reused existing selection: False
Core skills: ['Java', 'SQL', 'Git / Version Control', 'C++']
Changeable skills: ['Spring Boot', 'MySQL', 'CI/CD']
Unclassified skills: ['communication', 'problem solving', 'teamwork']


## Verify and display the six stored rows

In [4]:
from collections import Counter

errors = []
if len(result.questions) != 6:
    errors.append(f'expected 6 questions, received {len(result.questions)}')
if [question.question_id for question in result.questions] != list(dict.fromkeys(question.question_id for question in result.questions)):
    errors.append('duplicate Question Bank IDs')
distribution = Counter(question.difficulty for question in result.questions)
if distribution != Counter({'Easy': 3, 'Medium': 2, 'Hard': 1}):
    errors.append(f'invalid distribution: {dict(distribution)}')
if errors:
    raise RuntimeError('; '.join(errors))
for order, question in enumerate(result.questions, start=1):
    print(f'Q{order} [{question.difficulty}] {question.subject} / {question.topic}: {question.question}')
print('Validation: PASS (6 unique rows, orders 1-6, 3 Easy / 2 Medium / 1 Hard)')

Q1 [Easy] C++ / variables and types: In C++, What is variables and types, and what problem does it address in a batch job?
Q2 [Easy] C++ / variables and types: In C++, What is variables and types, and what problem does it address in a data-processing pipeline?
Q3 [Easy] Java / functions: In Java, What is functions, and what problem does it address in a multi-user application?
Q4 [Medium] C++ / functions: In C++, How should an engineer compare functions with its main alternative in a batch job?
Q5 [Hard] Java / variables and types: In Java, How would you apply variables and types responsibly when designing a system under test?
Q6 [Medium] Java / variables and types: In Java, A a command-line utility has a problem related to variables and types; what reasoning steps should diagnose it?
Validation: PASS (6 unique rows, orders 1-6, 3 Easy / 2 Medium / 1 Hard)
